In [0]:
USE CATALOG workspace;
USE SCHEMA new_default;



In [0]:
-- 0) Ausgangslage prüfen
SELECT * FROM managed_default;



country,code,dial_code
France,FR,+33
Spain,ES,+34


In [0]:
-- 1) SHALLOW CLONE anlegen
CREATE TABLE managed_default_clone
SHALLOW CLONE managed_default;



source_table_size,source_num_of_files,num_of_synced_transactions,num_removed_files,num_copied_files,removed_files_size,copied_files_size
1032,1,null,0,0,0,0


In [0]:
-- 2) Inhalt von Clone und Original vergleichen
SELECT 'ORIGINAL' AS src, * FROM managed_default
UNION ALL
SELECT 'CLONE'    AS src, * FROM managed_default_clone;



src,country,code,dial_code
ORIGINAL,France,FR,+33
ORIGINAL,Spain,ES,+34
CLONE,France,FR,+33
CLONE,Spain,ES,+34


In [0]:
-- 3) Änderungen NUR im CLONE vornehmen
INSERT INTO managed_default_clone VALUES ('Germany','DE','+49');
DELETE FROM managed_default_clone WHERE code = 'FR';



num_affected_rows
1


In [0]:
-- 4.1) Historie (Transaktionslog) anzeigen
DESCRIBE HISTORY managed_default;



version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2025-12-08T12:32:04.000Z,78230215195550,philippe.christen@fhnw.ch,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3539286768437899),1208-121211-pmhh4gic-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1032)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2025-12-08T12:32:03.000Z,78230215195550,philippe.christen@fhnw.ch,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.writePartitionColumnsToParquet"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-2efa921c-6eb8-48d1-92f1-d2bd3a21545a"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-303bd59c-6f96-4d3f-8cdf-812da2e62794""}, statsOnLoad -> false)",null,List(3539286768437899),1208-121211-pmhh4gic-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13


In [0]:
-- 4.2) Historie (Transaktionslog) anzeigen
DESCRIBE HISTORY managed_default_clone;


version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2025-12-08T12:36:37.000Z,78230215195550,philippe.christen@fhnw.ch,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3539286768437883),1208-121211-pmhh4gic-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2049, p25FileSize -> 1828, numDeletionVectorsRemoved -> 1, minFileSize -> 1828, numAddedFiles -> 1, maxFileSize -> 1828, p75FileSize -> 1828, p50FileSize -> 1828, numAddedBytes -> 1828)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
2,2025-12-08T12:36:35.000Z,78230215195550,philippe.christen@fhnw.ch,DELETE,"Map(predicate -> [""(code#16687 = FR)""])",null,List(3539286768437883),1208-121211-pmhh4gic-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1526, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1158, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 368)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
1,2025-12-08T12:36:32.000Z,78230215195550,philippe.christen@fhnw.ch,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3539286768437883),1208-121211-pmhh4gic-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1017)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2025-12-08T12:36:25.000Z,78230215195550,philippe.christen@fhnw.ch,CLONE,"Map(source -> workspace.new_default.managed_default, sourceVersion -> 1, isShallow -> true)",null,List(3539286768437883),1208-121211-pmhh4gic-v2n,-1,Serializable,false,"Map(removedFilesSize -> 0, numRemovedFiles -> 0, sourceTableSize -> 1032, numCopiedFiles -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, copiedFilesSize -> 0, sourceNumOfFiles -> 1)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13


In [0]:
-- 5) Nachweis: Original unverändert, Clone geändert
SELECT 'ORIGINAL' AS src, * FROM managed_default
UNION ALL
SELECT 'CLONE'    AS src, * FROM managed_default_clone;



src,country,code,dial_code
ORIGINAL,France,FR,+33
ORIGINAL,Spain,ES,+34
CLONE,Spain,ES,+34
CLONE,Germany,DE,+49


In [0]:
-- 6) Anmerkung: Würde das Original gelöscht werden, funktioniert der Clone weiter (Daten bleiben bestehen, bis sie explizit aus dem Storage entfernt oder via VACUUM bereinigt werden)
-- Die VACUUM-Funktion ist in der Free Edition nicht verfügbar.

-- DROP TABLE managed_default;
-- SELECT * FROM managed_default_clone order by country desc


In [0]:
-- aufräumen
DROP TABLE managed_default_clone;
DROP TABLE managed_default;
DROP SCHEMA new_default;